# Re-evaluate Baseline theo Code-Switching Subset - ViGoEmotions

Notebook này dùng để chạy trên Kaggle. Mục tiêu: đọc ViGoEmotions từ Kaggle Dataset trong `/kaggle/input`, gắn nhóm code-switching bằng rule-based detector, chạy inference bằng baseline đã train, rồi báo cáo metric theo từng subset: `pure_vi`, `code_switched`, `english_mixed`, `teencode_slang`, `emoji_only`, `other_noise`.

In [ ]:
!pip -q install datasets transformers accelerate evaluate scikit-learn

In [ ]:
import os
import re
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, hamming_loss
from tqdm.auto import tqdm

MODEL_NAME_OR_PATH = "/kaggle/input/YOUR_BASELINE_MODEL_PATH"
# Ví dụ:
# MODEL_NAME_OR_PATH = "/kaggle/input/xlm-r-vigoemotions/best_model"
# MODEL_NAME_OR_PATH = "/kaggle/working/vigo_baseline_outputs/xlm-r-vigoemotions/best_model"

# Sửa path này theo Kaggle Dataset bạn add vào notebook.
DATA_DIR = "/kaggle/input/YOUR_VIGOEMOTIONS_DATASET"

MAX_LENGTH = 128
BATCH_SIZE = 32
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABEL_NAMES = [
    "amusement", "excitement", "joy", "love", "desire", "optimism", "caring",
    "pride", "admiration", "gratitude", "relief", "approval", "realization",
    "surprise", "curiosity", "confusion", "fear", "nervousness", "remorse",
    "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger",
    "annoyance", "disapproval", "neutral"
]

NUM_LABELS = len(LABEL_NAMES)


## Load ViGoEmotions từ Kaggle Dataset

Trên Kaggle, add dataset ViGoEmotions vào notebook, rồi sửa `DATA_DIR` cho đúng folder trong `/kaggle/input/...`. Notebook hỗ trợ file riêng `train/val/test` hoặc một file chung có cột `split`/`set`.

In [ ]:
def normalize_split_name(split_name):
    if split_name in ["validation", "dev"]:
        return "val"
    return split_name

def read_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        xls = pd.ExcelFile(path)
        if {"train", "val", "test"}.issubset(set(xls.sheet_names)):
            frames = []
            for sheet in ["train", "val", "test"]:
                df = pd.read_excel(path, sheet_name=sheet)
                df["split"] = sheet
                frames.append(df)
            return pd.concat(frames, ignore_index=True)
        return pd.read_excel(path, sheet_name=xls.sheet_names[0])
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in [".json", ".jsonl"]:
        return pd.read_json(path, lines=(suffix == ".jsonl"))
    raise ValueError(f"Unsupported file type: {path}")

def find_data_files(data_dir):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"DATA_DIR does not exist: {data_dir}\n"
            "Set DATA_DIR to your Kaggle dataset folder, e.g. /kaggle/input/vigoemotions"
        )
    exts = {".csv", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"}
    files = [p for p in data_dir.rglob("*") if p.is_file() and p.suffix.lower() in exts]
    if not files:
        raise FileNotFoundError(f"No supported data files found under: {data_dir}")
    return files

def standardize_columns(df):
    df = df.copy()
    lower_map = {c.lower(): c for c in df.columns}

    if "text" not in df.columns:
        for candidate in ["comment", "sentence", "content", "clean_text"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "text"})
                break

    if "labels" not in df.columns:
        for candidate in ["label", "emotion", "emotions", "target", "targets"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "labels"})
                break

    if "split" not in df.columns:
        for candidate in ["set", "data_split"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "split"})
                break

    missing = [c for c in ["text", "labels"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Available columns: {list(df.columns)}")

    return df

files = find_data_files(DATA_DIR)
print("Found files:")
for f in files:
    print("-", f)

named_split_frames = []
single_frames = []

for file_path in files:
    df = standardize_columns(read_table(file_path))
    name = file_path.stem.lower()

    if "split" in df.columns:
        single_frames.append(df)
    elif "train" in name:
        df["split"] = "train"
        named_split_frames.append(df)
    elif any(k in name for k in ["val", "dev", "validation"]):
        df["split"] = "val"
        named_split_frames.append(df)
    elif "test" in name:
        df["split"] = "test"
        named_split_frames.append(df)

if single_frames:
    data = pd.concat(single_frames, ignore_index=True)
elif named_split_frames:
    data = pd.concat(named_split_frames, ignore_index=True)
else:
    raise ValueError(
        "Cannot infer splits. Use files named train/val/test, "
        "or provide a single file with column split/set."
    )

data = standardize_columns(data)
data["split"] = data["split"].astype(str).str.lower().map(normalize_split_name)

print(data.shape)
print(data["split"].value_counts())
data.head()


In [ ]:
required_splits = {"train", "val", "test"}
found_splits = set(data["split"].dropna().unique())
missing_splits = required_splits - found_splits
if missing_splits:
    raise ValueError(f"Missing splits: {missing_splits}. Found splits: {found_splits}")

data.head()


In [ ]:
def parse_labels(x, num_labels=NUM_LABELS):\n    if isinstance(x, np.ndarray):\n        x = x.tolist()\n\n    if isinstance(x, str):\n        try:\n            x = ast.literal_eval(x)\n        except Exception:\n            x = [int(i) for i in re.findall(r"\\d+", x)]\n\n    if isinstance(x, (list, tuple)):\n        x = list(x)\n\n        if len(x) == num_labels and set(np.unique(x)).issubset({0, 1, 0.0, 1.0}):\n            return np.array(x, dtype=int)\n\n        y = np.zeros(num_labels, dtype=int)\n        for idx in x:\n            if 0 <= int(idx) < num_labels:\n                y[int(idx)] = 1\n        return y\n\n    y = np.zeros(num_labels, dtype=int)\n    try:\n        y[int(x)] = 1\n    except Exception:\n        pass\n    return y\n\ndata["y_true"] = data["labels"].apply(parse_labels)

## Rule-Based Code-Switching Detector

In [ ]:
URL_RE = re.compile(r"https?://\\S+|www\\.\\S+")\nMENTION_RE = re.compile(r"@\\w+")\nHASHTAG_RE = re.compile(r"#\\w+")\nTOKEN_RE = re.compile(r"[A-Za-zÀ-ỹĐđ0-9_]+", re.UNICODE)\nVIET_CHARS_RE = re.compile(r"[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", re.IGNORECASE)\nEMOJI_RE = re.compile("[" "\\U0001F300-\\U0001F5FF" "\\U0001F600-\\U0001F64F" "\\U0001F680-\\U0001F6FF" "\\U0001F1E0-\\U0001F1FF" "\\U00002700-\\U000027BF" "\\U0001F900-\\U0001F9FF" "\\U00002600-\\U000026FF" "]+", flags=re.UNICODE)\n\nVI_STOPWORDS_ASCII = {\n    "anh", "em", "toi", "tao", "may", "ban", "minh", "chung", "nguoi",\n    "la", "thi", "ma", "va", "voi", "cho", "cua", "mot", "nhung", "cac",\n    "khong", "ko", "k", "duoc", "dc", "dang", "da", "se", "rat", "qua",\n    "nay", "kia", "do", "day", "roi", "nhe", "nha", "nua", "luon"\n}\n\nEN_COMMON = {\n    "video", "clip", "comment", "bro", "game", "like", "fan", "sorry",\n    "team", "thank", "thanks", "love", "idol", "live", "share", "post",\n    "status", "facebook", "youtube", "tiktok", "admin", "best", "good",\n    "bad", "ok", "okay", "hello", "hi", "bye", "cute", "hot", "cool",\n    "top", "new", "old", "real", "fake", "drama", "trend", "follow",\n    "view", "views", "sub", "subscribe", "stream", "online", "offline",\n    "music", "movie", "show", "style", "girl", "boy", "baby", "happy",\n    "sad", "angry", "wow", "please", "perfect", "legend"\n}\n\nTEENCODE = {\n    "ko", "k", "kh", "khum", "hong", "hok", "hông", "hem",\n    "dc", "đc", "đx", "dx", "j", "z", "dz", "r", "ùi", "ui",\n    "mn", "mng", "ae", "ny", "crush", "ib", "rep", "cmt",\n    "ad", "add", "acc", "avt", "stt", "tym", "thik", "thích",\n    "mik", "mk", "mjk", "bn", "b", "t", "m", "cx", "cũng",\n    "vs", "vcl", "vl", "vãi", "kk", "kkk", "haha", "hihi",\n    "hj", "hjhj", "hehe", "huhu", "lol", "lmao"\n}\n\ndef is_english_like(token):\n    t = token.lower()\n    if not re.fullmatch(r"[a-z]+", t):\n        return False\n    if len(t) <= 2:\n        return False\n    if t in VI_STOPWORDS_ASCII or t in TEENCODE:\n        return False\n    if t in EN_COMMON:\n        return True\n    english_suffixes = ("ing", "tion", "ment", "er", "or", "ly", "ed", "ous", "ive", "al")\n    return len(t) >= 5 and t.endswith(english_suffixes)\n\ndef analyze_text(text):\n    text = "" if pd.isna(text) else str(text)\n    tokens = TOKEN_RE.findall(text)\n    lower_tokens = [t.lower() for t in tokens]\n    english_tokens = [t for t in lower_tokens if is_english_like(t)]\n    teencode_tokens = [t for t in lower_tokens if t in TEENCODE]\n    num_tokens = max(len(tokens), 1)\n    has_english = len(english_tokens) > 0\n    has_teencode = len(teencode_tokens) > 0\n    has_emoji = bool(EMOJI_RE.search(text))\n    has_url = bool(URL_RE.search(text))\n    has_mention = bool(MENTION_RE.search(text))\n    has_hashtag = bool(HASHTAG_RE.search(text))\n\n    if has_english:\n        cs_group = "english_mixed"\n    elif has_teencode:\n        cs_group = "teencode_slang"\n    elif has_emoji:\n        cs_group = "emoji_only"\n    elif has_url or has_mention or has_hashtag:\n        cs_group = "other_noise"\n    else:\n        cs_group = "pure_vi"\n\n    return pd.Series({\n        "num_tokens": len(tokens),\n        "has_vietnamese_diacritic": bool(VIET_CHARS_RE.search(text)),\n        "has_english": has_english,\n        "english_count": len(english_tokens),\n        "english_ratio": len(english_tokens) / num_tokens,\n        "english_tokens": " ".join(english_tokens),\n        "has_teencode": has_teencode,\n        "teencode_count": len(teencode_tokens),\n        "teencode_ratio": len(teencode_tokens) / num_tokens,\n        "teencode_tokens": " ".join(teencode_tokens),\n        "has_emoji": has_emoji,\n        "has_url": has_url,\n        "has_mention": has_mention,\n        "has_hashtag": has_hashtag,\n        "is_code_switched": has_english or has_teencode,\n        "cs_group": cs_group\n    })\n\ncs_features = data["text"].apply(analyze_text)\ndata = pd.concat([data, cs_features], axis=1)\n\ndisplay(data["cs_group"].value_counts())\ndisplay(pd.crosstab(data["split"], data["cs_group"], margins=True))

## Load Baseline Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=False)\n\nmodel = AutoModelForSequenceClassification.from_pretrained(\n    MODEL_NAME_OR_PATH,\n    num_labels=NUM_LABELS,\n    problem_type="multi_label_classification",\n    ignore_mismatched_sizes=True\n)\n\nmodel.to(DEVICE)\nmodel.eval()

## Inference trên Test Set

In [ ]:
@torch.no_grad()\ndef predict_proba(texts):\n    all_probs = []\n\n    for i in tqdm(range(0, len(texts), BATCH_SIZE)):\n        batch_texts = texts[i:i + BATCH_SIZE]\n        enc = tokenizer(\n            batch_texts,\n            padding=True,\n            truncation=True,\n            max_length=MAX_LENGTH,\n            return_tensors="pt"\n        )\n        enc = {k: v.to(DEVICE) for k, v in enc.items()}\n        outputs = model(**enc)\n        probs = torch.sigmoid(outputs.logits).detach().cpu().numpy()\n        all_probs.append(probs)\n\n    return np.vstack(all_probs)\n\ntest_df = data[data["split"] == "test"].reset_index(drop=True)\ny_true = np.vstack(test_df["y_true"].values)\ny_prob = predict_proba(test_df["text"].astype(str).tolist())\ny_pred = (y_prob >= THRESHOLD).astype(int)\n\nprint(y_true.shape, y_pred.shape)

## Evaluate theo Subset

In [ ]:
def multilabel_metrics(y_true, y_pred):\n    return {\n        "n_samples": len(y_true),\n        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),\n        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),\n        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),\n        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),\n        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),\n        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),\n        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),\n        "hamming_loss": hamming_loss(y_true, y_pred),\n        "subset_accuracy": accuracy_score(y_true, y_pred),\n    }\n\ntest_eval = test_df.copy()\ntest_eval["row_id"] = np.arange(len(test_eval))\n\nsubsets = {\n    "all": np.ones(len(test_eval), dtype=bool),\n    "pure_vi": test_eval["cs_group"].eq("pure_vi").values,\n    "code_switched": test_eval["is_code_switched"].eq(True).values,\n    "english_mixed": test_eval["cs_group"].eq("english_mixed").values,\n    "teencode_slang": test_eval["cs_group"].eq("teencode_slang").values,\n    "emoji_only": test_eval["cs_group"].eq("emoji_only").values,\n    "other_noise": test_eval["cs_group"].eq("other_noise").values,\n}\n\nsubset_rows = []\n\nfor subset_name, mask in subsets.items():\n    if mask.sum() == 0:\n        continue\n    metrics = multilabel_metrics(y_true[mask], y_pred[mask])\n    metrics["subset"] = subset_name\n    metrics["ratio"] = mask.mean()\n    subset_rows.append(metrics)\n\nsubset_metrics_df = pd.DataFrame(subset_rows)\nsubset_metrics_df = subset_metrics_df[[\n    "subset", "n_samples", "ratio",\n    "micro_f1", "macro_f1", "weighted_f1",\n    "micro_precision", "macro_precision",\n    "micro_recall", "macro_recall",\n    "hamming_loss", "subset_accuracy"\n]]\n\ndisplay(subset_metrics_df)

In [ ]:
per_label_rows = []\n\nfor subset_name, mask in subsets.items():\n    if mask.sum() == 0:\n        continue\n\n    y_t = y_true[mask]\n    y_p = y_pred[mask]\n\n    f1_each = f1_score(y_t, y_p, average=None, zero_division=0)\n    precision_each = precision_score(y_t, y_p, average=None, zero_division=0)\n    recall_each = recall_score(y_t, y_p, average=None, zero_division=0)\n\n    for i, label in enumerate(LABEL_NAMES):\n        per_label_rows.append({\n            "subset": subset_name,\n            "label": label,\n            "support": int(y_t[:, i].sum()),\n            "precision": precision_each[i],\n            "recall": recall_each[i],\n            "f1": f1_each[i],\n        })\n\nper_label_metrics_df = pd.DataFrame(per_label_rows)\ndisplay(per_label_metrics_df.head())

In [ ]:
compare_df = subset_metrics_df[\n    subset_metrics_df["subset"].isin(["pure_vi", "code_switched", "english_mixed", "teencode_slang"])\n].copy()\n\ndisplay(compare_df[["subset", "n_samples", "micro_f1", "macro_f1", "hamming_loss"]])

## Save Outputs

In [ ]:
OUT_DIR = "/kaggle/working/baseline_subset_eval"\nos.makedirs(OUT_DIR, exist_ok=True)\n\nsubset_metrics_df.to_csv(f"{OUT_DIR}/subset_metrics.csv", index=False)\nper_label_metrics_df.to_csv(f"{OUT_DIR}/per_label_metrics_by_subset.csv", index=False)\n\ntest_eval_out = test_eval.drop(columns=["y_true"]).copy()\n\nfor i, label in enumerate(LABEL_NAMES):\n    test_eval_out[f"true_{label}"] = y_true[:, i]\n    test_eval_out[f"prob_{label}"] = y_prob[:, i]\n    test_eval_out[f"pred_{label}"] = y_pred[:, i]\n\ntest_eval_out.to_csv(f"{OUT_DIR}/test_predictions_with_code_switching.csv", index=False)\n\nprint("Saved to:", OUT_DIR)\nprint(os.listdir(OUT_DIR))

In [ ]:
display(subset_metrics_df)\n\ndisplay(\n    per_label_metrics_df\n    .sort_values(["subset", "f1"], ascending=[True, True])\n    .groupby("subset")\n    .head(10)\n)